In [6]:
import numpy as np
import pandas as pd
import sweetviz as sv # helps generate visualizations
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [3]:
insurance = pd.read_csv("insurance.csv")
insurance.head()

report = sv.analyze(insurance)
report.show_html("insurance_report.html")

Done! Use 'show' commands to display/save.   |██████████| [100%]   00:00 -> (00:00 left)


Report insurance_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


### SweetViz findings

The dataset shows us a mix of numeric variables such as age, bmi, children, and charges, along with categorical variables such as sex, smoker, and region. The strongest pattern is that smoking status is associated with much higher medical charges than non-smoking status.

Charges also tend to increase for people with higher BMI values; however, that relationship is not as strong as the smoker effect. Age seems to have a positive relationship with charges too, while sex and region appear to have weaker differences in comparison.

Therefore, we can say from EDA that smoker, age, and bmi are likely to be the most useful predictors of insurance charges.

In [4]:
insurance_encoded = pd.get_dummies(insurance, drop_first=True)

In [5]:
X = insurance_encoded.drop("charges", axis=1)
y = insurance_encoded["charges"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# we dont use fit transform on the test set because we want to use the same scaling as the training set


In [9]:
baseline_lr = LinearRegression()
baseline_lr.fit(X_train_scaled, y_train)

y_pred_lr = baseline_lr.predict(X_test_scaled)

mse_lr = mean_squared_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print(f"Baseline Linear Regression MSE: {mse_lr:.2f}")
print(f"Baseline Linear Regression R^2: {r2_lr:.2f}")

Baseline Linear Regression MSE: 33596915.85
Baseline Linear Regression R^2: 0.78


In [10]:
baseline_rf = RandomForestRegressor(random_state=42)
baseline_rf.fit(X_train_scaled, y_train)

y_pred_rf = baseline_rf.predict(X_test_scaled)

mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print(f"Baseline Random Forest MSE: {mse_rf:.2f}")
print(f"Baseline Random Forest R^2: {r2_rf:.2f}")

Baseline Random Forest MSE: 20997250.35
Baseline Random Forest R^2: 0.86


### Baseline Model Performance

The baseline linear regression provides a useful threshold/base finding because it assumes a more linear relationship between the predictors and insurance charges. But the random forest model is more flexible and can capture nonlinear relationships and interactions between these variables, which is porbably more realistic.

Based on the test metrics above, the stronger model is the random forest regressor. In this dataset, the RFR performs better because insurance charges are influenced by interactions and nonlinear effects, especially around smoking status and BMI.

In [11]:
# next step: cross validation on baseline models

cv_lr = cross_val_score(baseline_lr, X_train_scaled, y_train, cv=5, scoring="r2")
cv_rf = cross_val_score(baseline_rf, X_train_scaled, y_train, cv=5, scoring="r2")

print(f"Baseline Linear Regression CV R^2: {cv_lr.mean():.2f} ± {cv_lr.std():.2f}")
print(f"Baseline Random Forest CV R^2: {cv_rf.mean():.2f} ± {cv_rf.std():.2f}")

# why does it make sense that the models got "dumber" after cross validation?
# because we are training on less data in each fold, so the models have less information to learn from, 
# which can lead to worse performance on the test set.

Baseline Linear Regression CV R^2: 0.73 ± 0.05
Baseline Random Forest CV R^2: 0.82 ± 0.04


In [14]:
# next step: hyperparameter tuning on the random forest model using grid search cross validation
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "max_features": [1.0, "sqrt"]
}

grid_search_rf = GridSearchCV(
    estimator=baseline_rf,
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    return_train_score=True
 )
grid_search_rf.fit(X_train_scaled, y_train)

best_rf_params = grid_search_rf.best_params_
best_rf = grid_search_rf.best_estimator_

cv_results_rf = pd.DataFrame(grid_search_rf.cv_results_)
top_evals_rf = cv_results_rf.sort_values(
    by=["rank_test_score", "mean_test_score"],
    ascending=[True, False]
).head(20)[[
    "param_n_estimators",
    "param_max_depth",
    "param_min_samples_split",
    "param_max_features",
    "mean_train_score",
    "mean_test_score",
    "std_test_score",
    "rank_test_score"
]]

print("Top 20 Random Forest configurations:")
display(top_evals_rf)

print(f"Best Random Forest Parameters: {best_rf_params}")
print(f"Best Random Forest CV R^2: {grid_search_rf.best_score_:.4f}")

Top 20 Random Forest configurations:


,param_n_estimators,param_max_depth,param_min_samples_split,param_max_features,mean_train_score,mean_test_score,std_test_score,rank_test_score
11,200,10,5,1.0,0.947032,0.830343,0.042111,1
10,100,10,5,1.0,0.946334,0.830159,0.042303,2
3,200,None,5,1.0,0.951112,0.829550,0.042314,3
19,200,20,5,1.0,0.951111,0.829549,0.042317,4
2,100,None,5,1.0,0.950340,0.829511,0.042367,5
18,100,20,5,1.0,0.950339,0.829508,0.042371,6
9,200,10,2,1.0,0.967609,0.826878,0.040456,7
8,100,10,2,1.0,0.966753,0.826570,0.040521,8
17,200,20,2,1.0,0.975368,0.824374,0.040734,9
1,200,None,2,1.0,0.975374,0.824332,0.040720,10


Best Random Forest Parameters: {'max_depth': 10, 'max_features': 1.0, 'min_samples_split': 5, 'n_estimators': 200}
Best Random Forest CV R^2: 0.8303


### Top 20 Grid Search Models

From the top 20 results, the best-performing configuration is: `n_estimators=200`, `max_depth=10`, `min_samples_split=5`, and `max_features=1.0` with mean CV $R^2 \approx 0.8303$. The next few models are pretty close in performance, but this setting gives us the highest cross-validated score.

A useful pattern in the table is that the strongest models consistently use `min_samples_split=5` and `max_features=1.0`. Compared with `min_samples_split=2`, these settings have slightly lower training scores but better test scores, which suggests better generalization. I would choose the first ranking model because it has the best CV performance while keeping the train-test gap more controlled than its overly complex alternatives.

In [16]:
# compare training versus test performance for the tuned random forest
y_train_pred_tuned = best_rf.predict(X_train_scaled)
y_test_pred_tuned = best_rf.predict(X_test_scaled)

tuned_train_r2 = r2_score(y_train, y_train_pred_tuned)
tuned_test_r2 = r2_score(y_test, y_test_pred_tuned)
tuned_train_mse = mean_squared_error(y_train, y_train_pred_tuned)
tuned_test_mse = mean_squared_error(y_test, y_test_pred_tuned)

print(f"Tuned Random Forest Train R^2: {tuned_train_r2:.4f}")
print(f"Tuned Random Forest Test R^2: {tuned_test_r2:.4f}")
print(f"Tuned Random Forest Train MSE: {tuned_train_mse:.2f}")
print(f"Tuned Random Forest Test MSE: {tuned_test_mse:.2f}")

Tuned Random Forest Train R^2: 0.9452
Tuned Random Forest Test R^2: 0.8696
Tuned Random Forest Train MSE: 7915236.13
Tuned Random Forest Test MSE: 20251582.28


### Tuned Random Forest: Train vs Test

To assess overfitting, I compared training and test performance for the tuned Random Forest. The model achieved train $R^2 = 0.9452$ and test $R^2 = 0.8696$, with train MSE $= 7,915,236.13$ and test MSE $= 20,251,582.28$.

These results show some overfitting because the model performs better on the training set than on the test set. However, the test $R^2$ is still strong, so we can at least say that this is mild/moderate overfitting rather than severe overfitting. Overall, the tuned model generalizes reasonably well while still benefiting from the hyperparameter optimization.